<a href="https://colab.research.google.com/github/Leendew/IA_avanzada_Jesus/blob/main/EcoBicisEXCM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import zipfile
import io

In [ ]:
# Vamos a extraer los datos de viajes de Ecobici de un mes específico.
# Fuente: Portal de Datos Abiertos de la CDMX.
# Usaremos los datos de Julio de 2026 como ejemplo.
url= "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/08/public_data_web_2026-07.csv"
csv_file_name = "2026-07.csv"
print(f"Descargando datos desde: {url}")
try:
  response = requests.get(url, timeout=1200) # Added a timeout of 600 seconds (10 minutes)
  response.raise_for_status() # Raise an exception for bad status codes
  print ("Descarga completada con éxito.")

except requests.exceptions.Timeout as e:
  print(f"Error de timeout durante la descarga: {e}")
  df_raw = pd.DataFrame()
except requests.exceptions.RequestException as e:
  print(f"Error durante la descarga: {e})")
  df_raw = pd.DataFrame()

Descargando datos desde: https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/08/public_data_web_2026-07.csv


In [ ]:
# Save the zip file locally
with open(csv_file_name, 'wb') as f:
  f.write(response.content)
print(f"Archivo CSV guardado como {csv_file_name}")

# Read the extracted CSV into a DataFrame
print(f"Leyendo datos desde: {csv_file_name}")
df_raw = pd.read_csv(csv_file_name)
print("Extracción completada con éxito.")
print(f"Se cargaron {df_raw.shape[0]} registros.")

In [ ]:
# Mostrar el tamaño del DataFrame (filas y columnas)
print("Tamaño del DataFrame:")
print(df_raw. shape)

# Mostrar una previsualización de los datos
print("InPrevisualización del DataFrame:")
display(df_raw.head(80))

In [ ]:
# FASE 2: TRANSFORMACIÓN (Transform)

print("\n--- FASE 2: TRANSFORMACIÓN ---")

# Hacemos una copia para no modificar el DataFrame original
df = df_raw. copy()

# --- 2.1 Limpieza de Datos (Cleaning) ---
print ("\nIniciando limpieza de datos...")
# Convertir columnas de fecha a formato datetime
# Esto es crucial para poder realizar cálculos y extraer componentes.
df['Fecha_Retiro'] = pd.to_datetime(df['Fecha_Retiro'], dayfirst=True)
df['Fecha_Arribo'] = pd.to_datetime(df['Fecha_Arribo'], dayfirst=True)
print ("Columnas de fecha convertidas a datetime.")
display(df.head(10))

In [ ]:
# --- 2.2 Feature Engineering ---
print("\nIniciando Feature Engineering...")

# Para calcular la duración real, necesitamos combinar la fecha con la hora específica
df['Fecha_Retiro_Completa'] = pd.to_datetime(df['Fecha_Retiro'].dt.date.astype(str) + ' ' + df['Hora_Retiro'])
df['Fecha_Arribo_Completa'] = pd.to_datetime(df['Fecha_Arribo'].dt.date.astype(str) + ' ' + df['Hora_Arribo'])

# 1. Duración del viaje en minutos (usando las columnas completas)
df['duracion_minutos'] = (df['Fecha_Arribo_Completa'] - df['Fecha_Retiro_Completa']).dt.total_seconds() / 60

# 2. Día de la semana (0=Lunes, 6=Domingo)
df['dia_semana'] = df['Fecha_Retiro'].dt.dayofweek

# 3. Hora del día
df['hora_inicio'] = df['Fecha_Retiro_Completa'].dt.hour

# 4. Categoría de día (Fin de semana vs. Entre semana)
df['tipo_dia'] = df['dia_semana'].apply(lambda x: 'Fin de Semana' if x >= 5 else 'Entre Semana')

print("Nuevas características creadas con precisión: 'duracion_minutos', 'dia_semana', 'hora_inicio', 'tipo_dia'.")
display(df[['Fecha_Retiro_Completa', 'Fecha_Arribo_Completa', 'duracion_minutos']].head())



In [ ]:
# —- 2.3 Normalización / Estandarización —-
# Vamos a normalizar la, duración del viaje para que esté en una escala de 0 a 1.
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df['duracion_normalizada'] = scaler. fit_transform(df[['duracion_minutos']])
print("\n'duracion minutos' normalizada a una escala de 0 a 1.")
# —- 2.4 Encoding de Variables Categóricas —--
# La columna 'tipo_dia' es categórica. La convertiremos a números usando One-Hot Encoding.
df_encoded = pd.get_dummies(df, columns=['tipo_dia'], prefix='dia')
print("Variable 'tipo_dia' codificada con One-Hot Encoding.")
# -- 2.5 Balanceo de Clases —-
# Imaginemos que queremos predecir si un viaje es "muy largo" (> 60 min).
df['viaje_largo'] = df['duracion_minutos'] > 60
print("\nEjemplo de Balanceo de Clases:")
print("Distribución de 'viaje_largo' antes del balanceo:")
print(df['viaje_largo']. value_counts())
# —- Verificación del DataFrame Transformado —-print("\n--- Vista previa del DataFrame-transformado —__")
print(df_encoded.head ())